# Google Threat Intelligence lookup

Put your API key in `notebooks/GoogleThreatIntel/config.json` (copy `config.example.json` if needed). Then run the setup cell once, paste an indicator in the lookup cell, and run it.

Uses VirusTotal / Google TI v3. The `x-tool` header is required so `gti_assessment` is returned.


In [1]:
from __future__ import annotations

import base64
import ipaddress
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.parse import quote

import pandas as pd
import requests
from IPython.display import display, Markdown

GTI_BASE = "https://www.virustotal.com/api/v3"
X_TOOL = "htoc.VirusTotalApi.v0.1"
_HEX_HASH = re.compile(r"^[0-9a-fA-F]{32}$|^[0-9a-fA-F]{40}$|^[0-9a-fA-F]{64}$")


def repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for base in (here, *here.parents):
        if (base / "htoc_ml").is_dir():
            return base
    return here


ROOT = repo_root()
CONFIG_PATH = ROOT / "notebooks" / "GoogleThreatIntel" / "config.json"
EXAMPLE_CONFIG_PATH = CONFIG_PATH.with_name("config.example.json")


def load_gti_config(path: Path) -> dict[str, Any]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}. Copy {EXAMPLE_CONFIG_PATH} to config.json and add your API key."
        )
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"{path} must be a JSON object with an 'api_key' field.")
    return payload


GTI_CONFIG = load_gti_config(CONFIG_PATH)
API_KEY = str(GTI_CONFIG.get("api_key") or "").strip()
placeholder = "PASTE_YOUR_GOOGLE_THREAT_INTEL_API_KEY_HERE"
if not API_KEY or API_KEY == placeholder:
    raise RuntimeError(f"Set api_key in {CONFIG_PATH}.")

HEADERS = {
    "Accept": "application/json",
    "x-apikey": API_KEY,
    "x-tool": str(GTI_CONFIG.get("x_tool") or X_TOOL),
}


def classify_ioc(value: str) -> str:
    text = value.strip()
    if _HEX_HASH.fullmatch(text):
        return "file"
    if text.lower().startswith(("http://", "https://")):
        return "url"
    try:
        ipaddress.ip_address(text)
        return "ip"
    except ValueError:
        return "domain"


def url_id(url: str) -> str:
    return base64.urlsafe_b64encode(url.encode()).decode().rstrip("=")


def gti_path(value: str, ioc_type: str | None = None) -> str:
    kind = ioc_type or classify_ioc(value)
    if kind == "ip":
        return f"/ip_addresses/{quote(value, safe='')}"
    if kind == "domain":
        return f"/domains/{quote(value, safe='')}"
    if kind == "file":
        return f"/files/{quote(value, safe='')}"
    if kind == "url":
        return f"/urls/{url_id(value)}"
    raise ValueError(f"Unsupported IOC type: {kind}")


def gui_link(value: str, ioc_type: str) -> str:
    if ioc_type == "ip":
        return f"https://www.virustotal.com/gui/ip-address/{quote(value, safe='')}"
    if ioc_type == "domain":
        return f"https://www.virustotal.com/gui/domain/{quote(value, safe='')}"
    if ioc_type == "file":
        return f"https://www.virustotal.com/gui/file/{quote(value, safe='')}"
    return f"https://www.virustotal.com/gui/url/{url_id(value)}"


def gti_get(path: str, params: dict[str, Any] | None = None) -> dict[str, Any]:
    url = f"{GTI_BASE}{path}"
    response = requests.get(url, headers=HEADERS, params=params, timeout=30)
    if response.status_code == 404:
        raise FileNotFoundError(f"Not found in Google TI: {path}")
    if not response.ok:
        raise RuntimeError(f"GTI {response.status_code} {url}: {response.text[:500]}")
    return response.json()


def _nested(obj: Any, *keys: str) -> Any:
    cur = obj
    for key in keys:
        if not isinstance(cur, dict):
            return None
        cur = cur.get(key)
    return cur


def _unix_day(ts: Any) -> str | None:
    if ts in (None, ""):
        return None
    try:
        return datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime("%Y-%m-%d")
    except (TypeError, ValueError, OSError, OverflowError):
        return None


def pull_gti(indicator: str) -> dict[str, Any]:
    """Fetch Google TI assessment, detections, and related collections for one IOC."""
    value = str(indicator).strip()
    if not value:
        raise ValueError("indicator is empty")
    kind = classify_ioc(value)
    path = gti_path(value, kind)
    report = gti_get(path)
    data = report.get("data") or {}
    attrs = data.get("attributes") or {}
    stats = attrs.get("last_analysis_stats") or {}
    gti = attrs.get("gti_assessment") or {}
    threat = attrs.get("threat_severity") or {}
    votes = attrs.get("total_votes") or {}
    threat_class = attrs.get("popular_threat_classification") or {}

    detections = [
        {
            "engine": engine,
            "category": (row or {}).get("category"),
            "result": (row or {}).get("result"),
        }
        for engine, row in (attrs.get("last_analysis_results") or {}).items()
        if (row or {}).get("category") in {"malicious", "suspicious"}
    ]

    factors = []
    for name, val in (gti.get("contributing_factors") or {}).items():
        if val in (None, False, 0, "", [], {}):
            continue
        if isinstance(val, list):
            val = ", ".join(str(v) for v in val)
        factors.append({"factor": name, "value": val})

    cards = []
    for category, items in (gti.get("gti_description_cards") or {}).items():
        for card in items or []:
            cards.append(
                {
                    "category": category,
                    "influence": card.get("influence"),
                    "title": card.get("title"),
                    "detail": card.get("sub_title"),
                }
            )

    associations: list[dict[str, Any]] = []
    try:
        assoc = gti_get(f"{path}/associations", params={"limit": 20})
        for item in assoc.get("data") or []:
            a = item.get("attributes") or {}
            associations.append(
                {
                    "name": a.get("name"),
                    "collection_type": a.get("collection_type"),
                    "origin": a.get("origin"),
                    "id": item.get("id"),
                }
            )
    except Exception as exc:
        print(f"Related collections unavailable: {exc}")

    labels = threat_class.get("popular_threat_category") or []
    if isinstance(labels, list):
        labels = ", ".join(
            str(x.get("value") if isinstance(x, dict) else x) for x in labels
        )

    summary = {
        "indicator": value,
        "type": kind,
        "gti_verdict": _nested(gti, "verdict", "value"),
        "gti_severity": _nested(gti, "severity", "value"),
        "gti_threat_score": _nested(gti, "threat_score", "value"),
        "malicious": stats.get("malicious"),
        "suspicious": stats.get("suspicious"),
        "harmless": stats.get("harmless"),
        "undetected": stats.get("undetected"),
        "reputation": attrs.get("reputation"),
        "votes_malicious": votes.get("malicious"),
        "votes_harmless": votes.get("harmless"),
        "asn": attrs.get("asn"),
        "as_owner": attrs.get("as_owner"),
        "country": attrs.get("country"),
        "network": attrs.get("network"),
        "registrar": attrs.get("registrar"),
        "meaningful_name": attrs.get("meaningful_name"),
        "threat_label": threat_class.get("suggested_threat_label") or labels,
        "tags": ", ".join(attrs.get("tags") or []),
        "first_seen": _unix_day(attrs.get("first_seen_date") or attrs.get("first_submission_date")),
        "last_analysis": _unix_day(attrs.get("last_analysis_date")),
        "gui": gui_link(value, kind),
    }

    return {
        "summary": summary,
        "description": gti.get("description"),
        "contributing_factors": factors,
        "description_cards": cards,
        "detections": detections,
        "associations": associations,
        "report": report,
    }


def show_gti(info: dict[str, Any]) -> None:
    summary = info["summary"]
    display(Markdown(
        f"**{summary['indicator']}** ({summary['type']}) — "
        f"verdict `{summary['gti_verdict']}`, "
        f"severity `{summary['gti_severity']}`, "
        f"score `{summary['gti_threat_score']}`  "
        f"[Open in GTI]({summary['gui']})"
    ))
    if info.get("description"):
        display(Markdown(info["description"]))
    display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
    if info["contributing_factors"]:
        display(Markdown("#### Contributing factors"))
        display(pd.DataFrame(info["contributing_factors"]))
    if info["description_cards"]:
        display(Markdown("#### GTI explainability cards"))
        display(pd.DataFrame(info["description_cards"]))
    if info["associations"]:
        display(Markdown("#### Related GTI collections"))
        display(pd.DataFrame(info["associations"]))
    if info["detections"]:
        display(Markdown("#### Malicious / suspicious engines"))
        display(pd.DataFrame(info["detections"]))
    else:
        display(Markdown("_No malicious or suspicious engine detections._"))


print(f"Ready. Key loaded ({len(API_KEY)} chars). Paste an indicator in the next cell.")


Ready. Key loaded (64 chars). Paste an indicator in the next cell.


## Look up an indicator

Set `INDICATOR` to an IP, domain, URL, or MD5 / SHA-1 / SHA-256, then run this cell.


In [2]:
INDICATOR = "31.14.254.80"

gti_info = pull_gti(INDICATOR)
show_gti(gti_info)


**31.14.254.80** (ip) — verdict `VERDICT_UNDETECTED`, severity `SEVERITY_NONE`, score `1`  [Open in GTI](https://www.virustotal.com/gui/ip-address/31.14.254.80)

This indicator did not match our detection criteria and there is currently no evidence of malicious activity.

,value
indicator,31.14.254.80
type,ip
gti_verdict,VERDICT_UNDETECTED
gti_severity,SEVERITY_NONE
gti_threat_score,1
malicious,8
suspicious,5
harmless,46
undetected,30
reputation,-1


#### Contributing factors

,factor,value
0,safebrowsing_verdict,harmless
1,normalised_categories,phishing
2,gti_confidence_score,4


...

,category,influence,title,detail
0,MITIGATING_FACTORS,benign,Shared infrastructure,Verified Shared Infrastructure
1,MITIGATING_FACTORS,benign,High prevalence,High Global Prevalence
2,TECHNICAL_EVIDENCE,medium,Classification: phishing,
3,BULK_SIGNALS,medium,Multi-Engine AV Detections: 8,


#### Related GTI collections

,name,collection_type,origin,id
0,Honeypot IPs Collection,collection,Crowdsourced,5805b0570c4559c7be1baf1e93ab5fee720b5524a0e727...
1,Sonicwall-IM_Cust Inbound Malicious Communicat...,collection,Crowdsourced,10449a54d0de690efc448c71ecf9e5c5606c2989be9d6a...
2,Fortigate -ASA IM_Cust Inbound Malicious Commu...,collection,Crowdsourced,e3aa23d2a0f502f46a24012665a5ce0df6179510723d2c...
3,Ago-33,collection,Crowdsourced,1aff4e5e8acb053bfa44a3d5dbd091e38ceddcd31c5182...
4,IOC2,collection,Crowdsourced,1b58495a6bca2e3520e3f454784ac6e423057e03a4b1d5...
5,dejs jdshc,collection,Crowdsourced,20277dddab1189c7ffbc628710da410186e16fad5ffb98...
6,August IP batch -2,collection,Crowdsourced,8474c5876bcd0b9bd75f93552e407e3254704525601fae...
7,edj dejh,collection,Crowdsourced,8d1afab0530ec2666159849840bdd7963293f24a2376a9...


#### Malicious / suspicious engines

,engine,category,result
0,ADMINUSLabs,malicious,malicious
1,Criminal IP,malicious,malicious
2,alphaMountain.ai,suspicious,suspicious
3,AlphaSOC,suspicious,suspicious
4,BitDefender,malicious,phishing
5,CINS Army,malicious,malicious
6,CyRadar,suspicious,suspicious
7,Fortinet,malicious,malware
8,G-Data,malicious,phishing
9,Gridinsoft,suspicious,suspicious


## Optional: several indicators

Public keys are typically 4 requests per minute. Each indicator uses 2 calls (report + associations), so this cell waits between them.


In [ ]:
import time

INDICATORS = [
    # "1.2.3.4",
    # "evil.example",
]
PAUSE_S = 30.0

gti_rows = []
for i, ioc in enumerate(INDICATORS):
    try:
        info = pull_gti(ioc)
        gti_rows.append(info["summary"])
        show_gti(info)
    except Exception as exc:
        gti_rows.append({"indicator": ioc, "error": str(exc)})
        print(f"{ioc}: {exc}")
    if i < len(INDICATORS) - 1:
        time.sleep(PAUSE_S)

if gti_rows:
    display(pd.DataFrame(gti_rows))
